In [ ]:
!pip install qiskit qiskit-aer codecarbon numpy scipy matplotlib pandas -q

In [ ]:
import qiskit
import qiskit_aer
import codecarbon
import scipy
import numpy
import matplotlib

print("Qiskit version:", qiskit.__version__)
print("Qiskit Aer version:", qiskit_aer.__version__)
print("CodeCarbon version:", codecarbon.__version__)
print("SciPy version:", scipy.__version__)
print("All packages installed successfully! ✅")

In [ ]:
print(matplotlib.__version__)

In [ ]:
import sys
print(sys.version)

In [ ]:
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse.linalg import eigsh
from scipy.optimize import minimize
from codecarbon import EmissionsTracker

from qiskit.circuit.library import efficient_su2        # ✅ replaces TwoLocal
from qiskit.quantum_info import SparsePauliOp
from qiskit import transpile
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error
from qiskit_aer.primitives import EstimatorV2 as AerEstimator

print("All imports successful ✅")

In [ ]:
# H2 Hamiltonian in STO-3G basis (Jordan-Wigner mapping, 4 qubits)
# Ground state energy ≈ -1.8572 Hartree
def get_h2_hamiltonian():
    hamiltonian = SparsePauliOp.from_list([
        ("IIII", -0.8126),
        ("ZIII", 0.1712),
        ("IZII", -0.2228),
        ("IIZI", -0.2228),
        ("IIIZ", 0.1712),
        ("ZZII", 0.1686),
        ("IZIZ", 0.1200),
        ("ZIIZ", 0.1659),
        ("IZZI", 0.1659),
        ("ZIZI", 0.1200),
        ("IIZZ", 0.1686),
        ("XXYY", -0.0453),
        ("XYYX", 0.0453),
        ("YXXY", 0.0453),
        ("YYXX", -0.0453),
    ])
    return hamiltonian

hamiltonian = get_h2_hamiltonian()
print("H2 Hamiltonian defined ✅")
print(f"Number of qubits: {hamiltonian.num_qubits}")
print(f"Number of Pauli terms: {len(hamiltonian)}")

In [ ]:
def classical_solver(hamiltonian):
    """Solve H2 ground state classically using exact diagonalisation."""
    matrix = hamiltonian.to_matrix()

    tracker = EmissionsTracker(
        project_name="GQAT_Classical",
        log_level="error",
        save_to_file=False
    )

    tracker.start()
    start_time = time.time()

    # Exact diagonalisation
    eigenvalues, _ = eigsh(matrix, k=1, which='SA')
    ground_state_energy = eigenvalues[0]

    end_time = time.time()
    emissions_data = tracker.stop()

    classical_time = end_time - start_time
    # CodeCarbon returns kg CO2 — convert energy: kWh → Joules
    classical_energy_kwh = tracker._total_energy.kWh
    classical_energy_joules = classical_energy_kwh * 3_600_000

    print(f"✅ Classical solver complete")
    print(f"   Ground state energy : {ground_state_energy:.6f} Hartree")
    print(f"   Time                : {classical_time:.4f} s")
    print(f"   Energy consumed     : {classical_energy_joules:.6f} J")

    return {
        "ground_state_energy": ground_state_energy,
        "time": classical_time,
        "energy_joules": classical_energy_joules
    }

classical_results = classical_solver(hamiltonian)

In [ ]:
def build_noise_model():
    """Build a noise model approximating IBM Brisbane superconducting hardware."""
    noise_model = NoiseModel()

    # Single-qubit gate error (~0.05%)
    single_qubit_error = depolarizing_error(0.0005, 1)
    # Two-qubit gate error (~1%)
    two_qubit_error = depolarizing_error(0.01, 2)
    # Thermal relaxation (T1=150us, T2=100us, gate_time=50ns)
    thermal_error = thermal_relaxation_error(
        t1=150e3, t2=100e3, time=50
    )

    noise_model.add_all_qubit_quantum_error(single_qubit_error, ['u1', 'u2', 'u3', 'rx', 'ry', 'rz'])
    noise_model.add_all_qubit_quantum_error(two_qubit_error, ['cx', 'ecr'])
    noise_model.add_all_qubit_quantum_error(thermal_error, ['u1', 'u2', 'u3'])

    print("✅ Noise model built (IBM Brisbane approximation)")
    return noise_model

noise_model = build_noise_model()

In [ ]:
def run_vqe(hamiltonian, noise_model, shots, classical_energy):
    """Run VQE on Aer noise model — compatible with Qiskit 2.x / Aer 0.17.x"""

    num_qubits = hamiltonian.num_qubits

    # No flatten argument
    ansatz = efficient_su2(num_qubits, reps=2)

    # Correct AerEstimator V2 initialisation
    noisy_simulator = AerSimulator(noise_model=noise_model)
    estimator = AerEstimator(
        options={
            "backend_options": {"noise_model": noise_model},
            "run_options": {"shots": shots}
        }
    )

    # Transpile ansatz to AerSimulator target
    pm = generate_preset_pass_manager(
        optimization_level=1,
        backend=noisy_simulator
    )
    isa_ansatz = pm.run(ansatz)
    isa_hamiltonian = hamiltonian.apply_layout(isa_ansatz.layout)

    # Cost function
    cost_history = []
    def cost_function(params):
        job = estimator.run([(isa_ansatz, isa_hamiltonian, params)])
        value = float(job.result()[0].data.evs)
        cost_history.append(value)
        return value

    # Initial parameters
    np.random.seed(42)
    init_params = np.random.uniform(-np.pi, np.pi, isa_ansatz.num_parameters)

    # Track energy during VQE run
    tracker = EmissionsTracker(
        project_name=f"GQAT_VQE_shots_{shots}",
        log_level="error",
        save_to_file=False
    )

    tracker.start()
    start_time = time.time()

    result = minimize(
        cost_function,
        init_params,
        method='COBYLA',
        options={'maxiter': 200, 'rhobeg': 0.5}
    )

    end_time = time.time()
    tracker.stop()

    vqe_time          = end_time - start_time
    vqe_energy_kwh    = tracker._total_energy.kWh
    vqe_energy_j      = vqe_energy_kwh * 3_600_000
    vqe_ground_energy = result.fun
    fidelity_error    = abs(vqe_ground_energy - classical_energy)

    # Cryogenic amortisation — Layer 1 (fridge ~20kW) + Layer 2 (control ~3kW)
    cryo_power_w   = 23_000
    cryo_energy_j  = cryo_power_w * vqe_time
    total_q_energy = vqe_energy_j + cryo_energy_j

    print(f"\n✅ VQE complete (shots={shots})")
    print(f"   VQE energy estimate : {vqe_ground_energy:.6f} Hartree")
    print(f"   Fidelity error      : {fidelity_error:.6f} Hartree")
    print(f"   VQE runtime         : {vqe_time:.4f} s")
    print(f"   L3 energy (Colab)   : {vqe_energy_j:.8f} J")
    print(f"   L1+L2 cryo energy   : {cryo_energy_j:.4f} J")
    print(f"   Total quantum energy: {total_q_energy:.4f} J")
    print(f"   COBYLA iterations   : {len(cost_history)}")

    return {
        "shots":               shots,
        "vqe_energy_estimate": vqe_ground_energy,
        "fidelity_error":      fidelity_error,
        "vqe_time":            vqe_time,
        "l3_energy_joules":    vqe_energy_j,
        "cryo_energy_joules":  cryo_energy_j,
        "total_quantum_energy": total_q_energy,
        "iterations":          len(cost_history)
    }

print("VQE function defined ✅")

In [ ]:
shot_counts = [100, 500, 1000, 5000]
results = []

for shots in shot_counts:
    print(f"\n{'='*50}")
    print(f"Running VQE with {shots} shots...")
    r = run_vqe(
        hamiltonian,
        noise_model,
        shots,
        classical_results["ground_state_energy"]
    )
    results.append(r)

print("\n\n✅ All VQE runs complete!")

In [ ]:
def compute_gqat(vqe_result, classical_result):
    """Compute GQAT score from VQE and classical results."""
    Q_perf = 1 / vqe_result["vqe_time"]        # higher = faster
    C_perf = 1 / classical_result["time"]
    C_energy = classical_result["energy_joules"]
    Q_energy = vqe_result["total_quantum_energy"]

    gqat = (Q_perf / C_perf) * (C_energy / Q_energy)
    return gqat

# Build results table
rows = []
for r in results:
    gqat = compute_gqat(r, classical_results)
    rows.append({
        "Shots":            r["shots"],
        "Q_perf (s)":       round(r["vqe_time"], 4),
        "C_perf (s)":       round(classical_results["time"], 6),
        "Q_energy (J)":     round(r["total_quantum_energy"], 4),
        "C_energy (J)":     round(classical_results["energy_joules"], 6),
        "Fidelity Error":   round(r["fidelity_error"], 6),
        "GQAT Score":       round(gqat, 6)
    })

df = pd.DataFrame(rows)
print("\n========== GQAT RESULTS TABLE ==========")
print(df.to_string(index=False))

# Save to CSV
df.to_csv("gqat_results.csv", index=False)
print("\n✅ Results saved to gqat_results.csv")

In [ ]:
shots_list  = [r["shots"]      for r in results]
gqat_scores = [compute_gqat(r, classical_results) for r in results]
q_energies  = [r["total_quantum_energy"] for r in results]
fidelities  = [r["fidelity_error"]       for r in results]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("GQAT Experimental Results — H₂ VQE on IBM Brisbane Noise Model",
             fontsize=13, fontweight='bold')

# Plot 1 — GQAT Score vs Shots
axes[0].plot(shots_list, gqat_scores, 'o-', color='steelblue', linewidth=2, markersize=8)
axes[0].axhline(y=1.0, color='red', linestyle='--', linewidth=1.5, label='GQAT = 1 threshold')
axes[0].set_xlabel("Shot Count",        fontsize=11)
axes[0].set_ylabel("GQAT Score",        fontsize=11)
axes[0].set_title("GQAT Score vs Shot Count", fontsize=11)
axes[0].legend()
axes[0].set_xscale('log')
axes[0].grid(True, alpha=0.3)

# Plot 2 — Total Quantum Energy vs Shots
axes[1].bar([str(s) for s in shots_list], q_energies, color='coral', edgecolor='black')
axes[1].set_xlabel("Shot Count",              fontsize=11)
axes[1].set_ylabel("Total Quantum Energy (J)", fontsize=11)
axes[1].set_title("Quantum Energy vs Shot Count", fontsize=11)
axes[1].grid(True, alpha=0.3, axis='y')

# Plot 3 — Fidelity Error vs Shots
axes[2].plot(shots_list, fidelities, 's-', color='green', linewidth=2, markersize=8)
axes[2].set_xlabel("Shot Count",           fontsize=11)
axes[2].set_ylabel("Fidelity Error (Hartree)", fontsize=11)
axes[2].set_title("VQE Fidelity Error vs Shot Count", fontsize=11)
axes[2].set_xscale('log')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("gqat_figures.png", dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figures saved to gqat_figures.png")

In [ ]:
# ── Carbon Intensity & CO2 Data ──────────────────────────────────
import time
print("=" * 55)
print("CARBON INTENSITY & CO2 REPORT")
print("=" * 55)

co2_results = []

for r in results:
    shots = r["shots"]
    tracker = EmissionsTracker(
        project_name=f"CO2_shots_{shots}",
        log_level="error",
        save_to_file=False,
        country_2letter_iso_code="NG"   # ✅ Nigeria — fixed parameter
    )
    tracker.start()
    time.sleep(r["vqe_time"])
    tracker.stop()

    co2_kg  = tracker.final_emissions
    co2_g   = co2_kg * 1000
    energy_j = tracker._total_energy.kWh * 3_600_000

    co2_results.append({
        "shots": shots,
        "co2_grams": co2_g,
        "energy_j": energy_j
    })

    print(f"Shots={shots:>5}: CO2={co2_g:.6f} g | Energy={energy_j:.6f} J")

# Classical CO2
tracker_c = EmissionsTracker(
    log_level="error",
    save_to_file=False,
    country_2letter_iso_code="NG"   # ✅ fixed
)
tracker_c.start()
time.sleep(classical_results["time"])
tracker_c.stop()

classical_co2_g = tracker_c.final_emissions * 1000
print(f"\nClassical : CO2={classical_co2_g:.8f} g")
print(f"Country   : Nigeria (NG)")
print(f"Grid intensity applied automatically by CodeCarbon ✅")

In [ ]:
# ── Per-Job Amortisation Analysis ────────────────────────────────
print("=" * 55)
print("AMORTISATION SENSITIVITY ANALYSIS")
print("=" * 55)

jobs_per_day = 1000
fridge_power_w = 23_000
seconds_per_day = 86_400
daily_cryo_energy = fridge_power_w * seconds_per_day  # Joules/day
per_job_cryo = daily_cryo_energy / jobs_per_day

print(f"Daily cryogenic energy (23kW x 86400s): {daily_cryo_energy:,.0f} J")
print(f"Per-job amortisation ({jobs_per_day} jobs/day): {per_job_cryo:.2f} J")
print()
print(f"{'Shots':>6} | {'Per-sec Q_energy (J)':>22} | "
      f"{'Per-job Q_energy (J)':>22} | "
      f"{'GQAT (per-sec)':>16} | {'GQAT (per-job)':>14}")
print("-" * 90)

C_perf   = classical_results["time"]
C_energy = classical_results["energy_joules"]

for r in results:
    shots     = r["shots"]
    l3        = r["l3_energy_joules"]
    Q_perf    = r["vqe_time"]

    q_energy_per_sec = l3 + r["cryo_energy_joules"]
    q_energy_per_job = l3 + per_job_cryo

    gqat_per_sec = (Q_perf/C_perf) * (C_energy / q_energy_per_sec)
    gqat_per_job = (Q_perf/C_perf) * (C_energy / q_energy_per_job)

    print(f"{shots:>6} | {q_energy_per_sec:>22.2f} | "
          f"{q_energy_per_job:>22.4f} | "
          f"{gqat_per_sec:>16.8f} | {gqat_per_job:>14.6f}")

In [ ]:
# ── Fidelity-Weighted GQAT ────────────────────────────────────────
print("=" * 55)
print("FIDELITY-WEIGHTED GQAT (GQAT_F)")
print("=" * 55)
print("Formula: GQAT_F = GQAT x (1 - fidelity_error / E_classical)")
print()

E_classical = abs(classical_results["ground_state_energy"])  # 1.8572

for r in results:
    base_gqat    = compute_gqat(r, classical_results)
    fid_penalty  = r["fidelity_error"] / E_classical
    gqat_f       = base_gqat * (1 - fid_penalty)
    print(f"Shots={r['shots']:>5}: GQAT={base_gqat:.6f} | "
          f"Fid_penalty={fid_penalty:.6f} | GQAT_F={gqat_f:.6f}")

In [ ]:
# ── Benchmark Comparison Table ────────────────────────────────────
print("=" * 70)
print("QUANTUM BENCHMARK COMPARISON")
print("=" * 70)
print(f"{'Metric':<12} {'Measures Speed':^15} {'Measures Energy':^17} "
      f"{'Fidelity':^10} {'Reproducible':^14} {'Threshold':^10}")
print("-" * 70)
benchmarks = [
    ("QV",      "Yes", "No",  "Yes", "Yes", "No"),
    ("CLOPS",   "Yes", "No",  "No",  "Yes", "No"),
    ("XEB",     "Yes", "No",  "Yes", "No",  "No"),
    ("Green500","No",  "Yes", "No",  "Yes", "No"),
    ("GQAT",    "Yes", "Yes", "Yes", "Yes", "Yes"),
]
for b in benchmarks:
    print(f"{b[0]:<12} {b[1]:^15} {b[2]:^17} {b[3]:^10} "
          f"{b[4]:^14} {b[5]:^10}")
print()
print("GQAT is the only metric that combines all five properties.")